# COSC2673/COSC2793 Assessment 1 - Airbnb Price Regression

This notebook uses **Week 1-4 content only**:
- Supervised regression task definition
- EDA + preprocessing
- Linear Regression, Ridge, Lasso
- Hold-out validation + 5-fold cross-validation
- MAE, RMSE, R^2 evaluation

Run all cells top-to-bottom. Replace `student_number` in the final cell.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
sns.set_theme(style='whitegrid')


In [ ]:
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
display(train_df.head())

display(train_df.describe(include='all').T)
print('Missing values (train):')
display(train_df.isna().sum()[train_df.isna().sum() > 0])


## Task and Framework
- Task: predict `price` for Airbnb listings (supervised regression).
- Split: 80/20 train/validation (hold-out).
- Stability estimate: 5-fold CV on training split.
- Metrics: MAE, RMSE, R^2.
- Final test file stays untouched until final model selection.


In [ ]:
X = train_df.drop(columns=['price'])
y = train_df['price']

categorical_features = X.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
numeric_features = [c for c in X.columns if c not in categorical_features]

print('Categorical:', categorical_features)
print('Numeric:', numeric_features)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(y, kde=True, ax=axes[0])
axes[0].set_title('Price distribution')
sns.boxplot(x=y, ax=axes[1])
axes[1].set_title('Price boxplot')
plt.tight_layout()
plt.show()

corr = train_df[numeric_features + ['price']].corr(numeric_only=True)
plt.figure(figsize=(10, 6))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Numeric correlation heatmap')
plt.tight_layout()
plt.show()


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
    ]
)

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def metrics(y_true, y_pred):
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'R2': r2_score(y_true, y_pred)
    }

def evaluate(name, model):
    pipe = Pipeline([('preprocess', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)

    val_pred = pipe.predict(X_val)
    val_m = metrics(y_val, val_pred)

    cv = cross_validate(
        pipe, X_train, y_train, cv=kfold,
        scoring=('neg_mean_absolute_error', 'neg_root_mean_squared_error', 'r2'),
        n_jobs=-1
    )

    return {
        'Model': name,
        'Val_MAE': val_m['MAE'],
        'Val_RMSE': val_m['RMSE'],
        'Val_R2': val_m['R2'],
        'CV_MAE': -cv['test_neg_mean_absolute_error'].mean(),
        'CV_RMSE': -cv['test_neg_root_mean_squared_error'].mean(),
        'CV_R2': cv['test_r2'].mean()
    }, pipe


In [ ]:
# Model 1: Linear Regression
linear_row, linear_pipe = evaluate('LinearRegression', LinearRegression())
display(pd.DataFrame([linear_row]))


In [ ]:
# Model 2: Ridge (tune alpha)
ridge_alphas = np.logspace(-3, 3, 13)
ridge_rows = []

for a in ridge_alphas:
    row, _ = evaluate(f'Ridge(alpha={a:.4g})', Ridge(alpha=float(a), random_state=RANDOM_STATE))
    row['alpha'] = a
    ridge_rows.append(row)

ridge_df = pd.DataFrame(ridge_rows).sort_values('Val_RMSE').reset_index(drop=True)
best_ridge_alpha = float(ridge_df.loc[0, 'alpha'])
display(ridge_df.head())

plt.figure(figsize=(6, 4))
plt.plot(ridge_df['alpha'], ridge_df['Val_RMSE'], marker='o')
plt.xscale('log')
plt.title('Ridge alpha vs Validation RMSE')
plt.xlabel('alpha')
plt.ylabel('Val RMSE')
plt.tight_layout()
plt.show()


In [ ]:
# Model 3: Lasso (tune alpha)
lasso_alphas = np.logspace(-4, 1, 12)
lasso_rows = []

for a in lasso_alphas:
    row, _ = evaluate(f'Lasso(alpha={a:.4g})', Lasso(alpha=float(a), random_state=RANDOM_STATE, max_iter=20000))
    row['alpha'] = a
    lasso_rows.append(row)

lasso_df = pd.DataFrame(lasso_rows).sort_values('Val_RMSE').reset_index(drop=True)
best_lasso_alpha = float(lasso_df.loc[0, 'alpha'])
display(lasso_df.head())

plt.figure(figsize=(6, 4))
plt.plot(lasso_df['alpha'], lasso_df['Val_RMSE'], marker='o', color='orange')
plt.xscale('log')
plt.title('Lasso alpha vs Validation RMSE')
plt.xlabel('alpha')
plt.ylabel('Val RMSE')
plt.tight_layout()
plt.show()


In [ ]:
comparison = pd.DataFrame([
    linear_row,
    ridge_df.loc[0].to_dict(),
    lasso_df.loc[0].to_dict()
]).sort_values('Val_RMSE').reset_index(drop=True)

display(comparison[['Model', 'Val_MAE', 'Val_RMSE', 'Val_R2', 'CV_MAE', 'CV_RMSE', 'CV_R2']])
best_model_name = comparison.loc[0, 'Model']
print('Selected model:', best_model_name)

if best_model_name.startswith('LinearRegression'):
    final_estimator = LinearRegression()
elif best_model_name.startswith('Ridge'):
    final_estimator = Ridge(alpha=best_ridge_alpha, random_state=RANDOM_STATE)
else:
    final_estimator = Lasso(alpha=best_lasso_alpha, random_state=RANDOM_STATE, max_iter=20000)

final_pipeline = Pipeline([('preprocess', preprocessor), ('model', final_estimator)])
final_pipeline.fit(X, y)
test_pred = np.clip(final_pipeline.predict(test_df), a_min=0, a_max=None)

student_number = 's1234567'  # replace with your student number
out_file = f'{student_number}_predictions.csv'
pd.DataFrame({'price': test_pred}).to_csv(out_file, index=False)
print('Saved prediction file:', out_file)
display(pd.DataFrame({'price': test_pred}).head())


## Report Notes
Use outputs above to discuss:
1. Task definition and why regression is appropriate.
2. Dataset size, feature types, and distributions.
3. Correlations and feature usefulness.
4. Preprocessing choices (imputation, one-hot encoding, scaling).
5. Model comparison and common error patterns.
6. Hyper-parameter impact (alpha/lambda for Ridge or Lasso).
7. Ethical considerations: fairness, transparency, potential bias, responsible deployment.
